# Build processed train/val/test splits

Flattens HC3's reddit_eli5 subset into a binary-labeled (0=human, 1=chatgpt) dataframe, in two variants:

- **length_controlled**: pairs truncated, at sentence boundaries, to an approximately equal word count (main dataset)
- **raw**: original lengths (kept for the length-shortcut ablation)

Splits are grouped by question (`pair_id`) so a question's human and chatgpt answers always land in the same split. Output goes to `data/processed/{variant}/{train,val,test}.parquet`.

Truncation is sentence-boundary-aware, not a hard word-count cutoff — an earlier version truncated mid-sentence, which left "does this text end abruptly" as a shortcut nearly as strong as the raw-length one it was meant to remove (see the sanity check below).

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / 'src'))

from data_utils import load_hc3, flatten_to_binary, group_train_val_test_split

PROCESSED_DIR = Path.cwd().parent / 'data' / 'processed'
DOMAINS = ('reddit_eli5',)

raw_df = load_hc3()
raw_df.shape

(24322, 5)

In [2]:
def build_and_save(raw_df, length_control, tag):
    flat = flatten_to_binary(raw_df, domains=DOMAINS, length_control=length_control)
    train_df, val_df, test_df = group_train_val_test_split(flat)

    out_dir = PROCESSED_DIR / tag
    out_dir.mkdir(parents=True, exist_ok=True)
    train_df.to_parquet(out_dir / 'train.parquet', index=False)
    val_df.to_parquet(out_dir / 'val.parquet', index=False)
    test_df.to_parquet(out_dir / 'test.parquet', index=False)

    print(f"[{tag}] total={flat.shape[0]} train={train_df.shape[0]} "
          f"val={val_df.shape[0]} test={test_df.shape[0]}")
    print(f"[{tag}] label balance (train): {train_df['label'].value_counts().to_dict()}")
    return train_df, val_df, test_df


lc_train, lc_val, lc_test = build_and_save(raw_df, length_control=True, tag='length_controlled')

[length_controlled] total=33314 train=23318 val=4998 test=4998
[length_controlled] label balance (train): {1: 11659, 0: 11659}


In [3]:
raw_train, raw_val, raw_test = build_and_save(raw_df, length_control=False, tag='raw')

[raw] total=33314 train=23318 val=4998 test=4998
[raw] label balance (train): {1: 11659, 0: 11659}


## Sanity check: length control actually removed the length gap

In [4]:
import pandas as pd

for name, df in [('length_controlled', lc_train), ('raw', raw_train)]:
    print(f"--- {name} ---")
    print(df.groupby('label')['word_count'].describe()[['mean', 'std', 'min', 'max']])
    print()

--- length_controlled ---
            mean        std  min    max
label                                  
0      94.256969  54.407831  4.0  329.0
1      90.261343  58.268789  1.0  200.0

--- raw ---
             mean         std  min     max
label                                     
0      152.495325  196.541444  8.0  7904.0
1      174.829488   53.716141  8.0   639.0



In [5]:
def ends_clean(text):
    return text.strip().endswith(('.', '!', '?', '"'))


lc_train['ends_clean'] = lc_train['text'].apply(ends_clean)
print('sentence-boundary-aware truncation check (higher = better, want both labels close together):')
print(lc_train.groupby('label')['ends_clean'].mean())

sentence-boundary-aware truncation check (higher = better, want both labels close together):
label
0    0.918947
1    0.997170
Name: ends_clean, dtype: float64
